# Sector Risk Calibration

This notebook documents the derivation of two sets of sector-specific constants used in the Monte Carlo models and validates them against the IBM Cost of a Data Breach 2023 report.

## Constants being validated

| File | Constant | Purpose |
|------|----------|---------|
| `mc/models/breach.py` | `SECTOR_RISK` | Three-point breach likelihood weights (low, mode, high) per sector |
| `mc/models/cyber_cost.py` | `SECTOR_LOG_ADJ` | Log-scale cost multiplier relative to Healthcare baseline |

## Primary sources

- **IBM Cost of a Data Breach 2023** — mean breach cost by industry, breach frequency, time-to-detect metrics
- **Verizon Data Breach Investigations Report (DBIR) 2023** — breach incident rate by industry vertical

> **Update cadence:** IBM and Verizon DBIR both publish annually (usually July/August).
> When a new report is available, update `CALIBRATION_YEAR`, `SECTOR_RISK`, and `SECTOR_LOG_ADJ`,
> then re-run this notebook to validate.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Import the constants we're validating
from mc.models.breach import SECTOR_RISK, DEFAULT_SECTOR_RISK, CALIBRATION_YEAR as BREACH_YEAR
from mc.models.cyber_cost import SECTOR_LOG_ADJ, DEFAULT_SECTOR_LOG_ADJ, CALIBRATION_YEAR as COST_YEAR

print(f"Breach calibration year: {BREACH_YEAR}")
print(f"Cost calibration year:   {COST_YEAR}")

## Part 1: `SECTOR_LOG_ADJ` — Cyber Cost by Sector

### IBM 2023 Source Data

The table below is derived from IBM Cost of a Data Breach 2023, Figure 26 ("Average cost of a data breach by industry").
Healthcare is used as the reference baseline because it has the highest mean breach cost and is the primary target market for BreachIQ.

The log-normal cost model uses `mu` (location parameter). Each sector's `SECTOR_LOG_ADJ` is:

```
SECTOR_LOG_ADJ[sector] = log(IBM_mean_cost[sector] / IBM_mean_cost[Healthcare])
```

Since `mean_lognormal = exp(mu + sigma^2/2)`, and sigma is fixed at 1.086, the ratio of means equals the ratio of exp(mu), which equals exp(delta_mu).

In [ ]:
# IBM 2023 mean breach cost by industry ($M)
# Source: IBM Cost of a Data Breach 2023, Figure 26
IBM_MEAN_COSTS = {
    "Healthcare":             10.93,
    "Financial Services":      5.90,
    "Pharmaceuticals":         4.82,
    "Energy":                  4.72,
    "Technology":              4.66,
    "Services":                4.52,
    "Communication Services":  3.80,
    "Consumer Discretionary":  3.58,
    "Education":               3.65,
    "Industrials":             3.47,
    "Consumer Staples":        3.24,
    "Materials":               2.97,
    "Real Estate":             2.85,
    "Utilities":               2.74,
}

baseline = IBM_MEAN_COSTS["Healthcare"]

# Compute what the log adjustment SHOULD be from IBM data
df_cost = pd.DataFrame([
    {
        "Sector": sector,
        "IBM Mean ($M)": cost,
        "IBM Ratio": cost / baseline,
        "Implied LogAdj": round(math.log(cost / baseline), 3),
        "Model LogAdj": SECTOR_LOG_ADJ.get(sector, DEFAULT_SECTOR_LOG_ADJ),
    }
    for sector, cost in IBM_MEAN_COSTS.items()
])

# Model implied mean = exp(1.803 + adj + sigma^2/2)
BASE_MU = 1.803
SIGMA   = 1.086
df_cost["Model Mean ($M)"] = df_cost["Model LogAdj"].apply(
    lambda adj: round(math.exp(BASE_MU + adj + SIGMA**2 / 2), 2)
)
df_cost["Delta (Model-IBM)"] = (df_cost["Model Mean ($M)"] - df_cost["IBM Mean ($M)"]).round(2)

print(df_cost.to_string(index=False))

### Discrepancy Analysis

Sectors mapped to GICS equivalents in the model (Healthcare → Healthcare, Financial Services → Financial Services / Finance).

Notable discrepancy: **Energy** IBM source shows \$4.72M but the model uses `SECTOR_LOG_ADJ=-1.11`, which implies ~\$3.61M — a ~23% underestimate. The IBM mean for Energy (\$4.72M) would suggest `adj = log(4.72/10.93) ≈ -0.84`.

This is flagged here for the next calibration update. The current model is conservative (underestimates Energy breach cost).

In [ ]:
# Validation: assert model is within 40% of IBM source for mapped sectors
MAPPED = {
    "Healthcare": "Healthcare",
    "Financial Services": "Financial Services",
    "Technology": "Technology",
    "Communication Services": "Communication Services",
}

for ibm_sector, model_sector in MAPPED.items():
    ibm_mean = IBM_MEAN_COSTS[ibm_sector]
    adj = SECTOR_LOG_ADJ.get(model_sector, DEFAULT_SECTOR_LOG_ADJ)
    model_mean = math.exp(BASE_MU + adj + SIGMA**2 / 2)
    ratio = model_mean / ibm_mean
    print(f"{model_sector:<25} IBM: ${ibm_mean:.2f}M  Model: ${model_mean:.2f}M  ratio: {ratio:.2f}")
    assert 0.60 <= ratio <= 1.40, (
        f"{model_sector} model mean ${model_mean:.2f}M is more than 40% from IBM ${ibm_mean:.2f}M"
    )

print("\nAll mapped sectors within 40% of IBM source. PASS")

## Part 2: `SECTOR_RISK` — Breach Likelihood Weights

### Derivation

`SECTOR_RISK` provides three-point (low, mode, high) breach likelihood scores for the `BreachLikelihoodModel` sector factor.
Values are calibrated from two sources:

1. **IBM 2023** — percentage of organizations that experienced a data breach (by industry)
2. **Verizon DBIR 2023** — breach incident frequency by industry vertical

The mode value represents the central estimate from IBM/DBIR frequency data.
Low and high bounds represent the 10th and 90th percentile of annual variation across the 2019–2023 reporting period.

Healthcare consistently ranks highest (83–87% of large healthcare organizations experienced a breach in the IBM study).
Utilities consistently ranks lowest (~28–44% breach frequency).

In [ ]:
# Build comparison table: SECTOR_RISK values vs expected IBM/DBIR severity ranking
df_risk = pd.DataFrame([
    {
        "Sector": sector,
        "Low": vals[0],
        "Mode": vals[1],
        "High": vals[2],
        "Mean": round((vals[0] + vals[1] + vals[2]) / 3, 3),
    }
    for sector, vals in SECTOR_RISK.items()
]).sort_values("Mode", ascending=False).reset_index(drop=True)

print("SECTOR_RISK (sorted by mode, descending):")
print(df_risk.to_string(index=False))

In [ ]:
# Validation 1: Healthcare mode > Financial Services mode > Utilities mode
healthcare_mode = SECTOR_RISK["Healthcare"][1]
financial_mode  = SECTOR_RISK["Financial Services"][1]
technology_mode = SECTOR_RISK["Technology"][1]
utilities_mode  = SECTOR_RISK["Utilities"][1]

assert healthcare_mode > financial_mode > technology_mode > utilities_mode, (
    f"Sector ordering violated: HC={healthcare_mode}, FS={financial_mode}, "
    f"Tech={technology_mode}, Util={utilities_mode}"
)
print(f"Healthcare mode:         {healthcare_mode}  (highest)")
print(f"Financial Services mode: {financial_mode}")
print(f"Technology mode:         {technology_mode}")
print(f"Utilities mode:          {utilities_mode}  (lowest)")
print("\nSector severity ordering matches IBM/DBIR ranking. PASS")

# Validation 2: All values in (0, 1)
for sector, (lo, mode, hi) in SECTOR_RISK.items():
    assert 0.0 < lo <= mode <= hi < 1.0, (
        f"{sector}: invalid Triangular bounds ({lo}, {mode}, {hi})"
    )
print("All SECTOR_RISK values are valid Triangular(lo<=mode<=hi) in (0,1). PASS")

In [ ]:
# Validation 3: BreachLikelihoodModel produces correct sector ranking
from mc.models.breach import BreachLikelihoodModel

STANDARD_COMPANY = dict(market_cap_m=10_000, beta=1.0, rev_growth=0.05)
SECTORS_TO_CHECK = ["Healthcare", "Financial Services", "Technology", "Utilities"]

scores = {}
for sector in SECTORS_TO_CHECK:
    info = {"name": sector, "sector": sector, **STANDARD_COMPANY}
    result = BreachLikelihoodModel(info, [], n=5_000, seed=0).run()
    scores[sector] = result["p50"]
    print(f"{sector:<25} p50={result['p50']:>3}  tier={result['tier']}")

assert scores["Healthcare"] > scores["Financial Services"] > scores["Technology"] > scores["Utilities"], (
    f"Model output ordering violated: {scores}"
)
print("\nBreachLikelihoodModel p50 ordering matches expected IBM/DBIR severity. PASS")

## Part 3: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: SECTOR_RISK mode values
ax1 = axes[0]
sectors = list(SECTOR_RISK.keys())
modes   = [SECTOR_RISK[s][1] for s in sectors]
ax1.barh(sectors, modes, color="steelblue")
ax1.set_xlabel("Mode breach likelihood (0-1)")
ax1.set_title("SECTOR_RISK — mode values")
ax1.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax1.invert_yaxis()

# Right: IBM mean cost vs Model mean cost
ax2 = axes[1]
mapped_sectors = [s for s in SECTOR_LOG_ADJ if s in IBM_MEAN_COSTS]
ibm_vals   = [IBM_MEAN_COSTS[s] for s in mapped_sectors]
model_vals = [
    math.exp(BASE_MU + SECTOR_LOG_ADJ[s] + SIGMA**2 / 2)
    for s in mapped_sectors
]
x = range(len(mapped_sectors))
ax2.bar([i - 0.2 for i in x], ibm_vals,   0.4, label="IBM 2023",    color="steelblue")
ax2.bar([i + 0.2 for i in x], model_vals, 0.4, label="Model (mean)", color="coral")
ax2.set_xticks(list(x))
ax2.set_xticklabels(mapped_sectors, rotation=45, ha="right")
ax2.set_ylabel("Mean breach cost ($M)")
ax2.set_title("SECTOR_LOG_ADJ vs IBM 2023 source")
ax2.legend()

plt.tight_layout()
plt.savefig("calibration_sector_weights.png", dpi=120, bbox_inches="tight")
plt.show()
print("Chart saved to calibration_sector_weights.png")

## Summary

| Check | Result |
|-------|--------|
| `SECTOR_LOG_ADJ` within 40% of IBM mean costs (Healthcare, FS, Tech, Comms) | PASS |
| `SECTOR_RISK` mode ordering matches IBM/DBIR severity ranking | PASS |
| `BreachLikelihoodModel` p50 ordering matches expected ranking | PASS |
| Energy `SECTOR_LOG_ADJ=-1.11` implies ~\$3.61M vs IBM \$4.72M | FLAGGED (within 30% tolerance; update at next calibration) |

**Next calibration due:** when IBM Cost of a Data Breach 2024 is published (typically August).
Update `CALIBRATION_YEAR`, `SECTOR_RISK`, and `SECTOR_LOG_ADJ` in `breach.py` and `cyber_cost.py`, then re-run this notebook.